# Filter CSVs to strict 5-minute incoming cadence

For every `*.csv` under the input folder: sort by `TIMESTAMP`, keep a row only if the time since the **previous** row is about **300 s ± tolerance** (same rule as the plotting notebook). Write filtered tables to **`data_5mins_frequency/`** next to `data/` (same filenames for top-level CSVs).

A run log is saved as **`data_5mins_frequency/_filter_summary.csv`**. Adjust `DATA_DIR` / `REPO_ROOT` in the next cell if your CSVs live elsewhere (e.g. another `data` folder on the desktop).

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

# Repository root (change if you open the notebook from elsewhere).
REPO_ROOT = Path.cwd()
DATA_DIR = REPO_ROOT / "data"
OUT_DIR = REPO_ROOT / "data_5mins_frequency"

TIME_COL = "TIMESTAMP"
STEP_SECONDS = 300.0
STEP_TOL_SECONDS = 30.0

# If True, include CSVs in subfolders; output names use relative path with "__"
# (e.g. sub__file.csv). If False, only DATA_DIR/*.csv.
RECURSIVE = False

In [2]:
def list_csv_files(data_dir: Path, *, recursive: bool) -> list[Path]:
    data_dir = Path(data_dir)
    if not data_dir.is_dir():
        raise FileNotFoundError(f"Not a directory: {data_dir}")
    if recursive:
        return sorted(data_dir.rglob("*.csv"))
    return sorted(data_dir.glob("*.csv"))


def mask_incoming_five_minutes(ts: pd.Series, nominal_s: float, tol_s: float) -> np.ndarray:
    """Row i True iff Δt from previous row is nominal_s ± tol_s (row 0 iff first interval matches)."""
    dt_sec = ts.diff().dt.total_seconds().to_numpy()
    step_ok = np.zeros(len(ts), dtype=bool)
    step_ok[1:] = np.abs(dt_sec[1:] - float(nominal_s)) <= float(tol_s)
    keep = np.zeros(len(ts), dtype=bool)
    n = len(keep)
    if n == 0:
        return keep
    keep[0] = n > 1 and bool(step_ok[1])
    if n > 1:
        keep[1:] = step_ok[1:]
    return keep


def filter_csv_to_5min(path: Path, *, time_col: str, nominal_s: float, tol_s: float) -> tuple[pd.DataFrame, int, int]:
    df = pd.read_csv(path)
    if time_col not in df.columns:
        raise ValueError(f"{path.name}: missing column {time_col!r}. Columns: {list(df.columns)}")
    df = df.copy()
    df[time_col] = pd.to_datetime(df[time_col], dayfirst=True, format="mixed")
    df = df.sort_values(time_col).reset_index(drop=True)
    n_before = len(df)
    keep = mask_incoming_five_minutes(df[time_col], nominal_s, tol_s)
    out = df.loc[keep].reset_index(drop=True)
    return out, n_before, len(out)

In [4]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

csv_paths = list_csv_files(DATA_DIR, recursive=RECURSIVE)
if not csv_paths:
    raise FileNotFoundError(f"No CSV files found under {DATA_DIR} (recursive={RECURSIVE}).")

summary_rows = []
for path in csv_paths:
    out_name = path.name
    if RECURSIVE and path.parent != Path(DATA_DIR).resolve():
        # Avoid overwriting: prefix with parent folder name if nested
        rel = path.relative_to(DATA_DIR.resolve())
        safe_stem = "__".join(rel.with_suffix("").parts)
        out_name = safe_stem + ".csv"
    try:
        filtered, n_in, n_out = filter_csv_to_5min(
            path,
            time_col=TIME_COL,
            nominal_s=STEP_SECONDS,
            tol_s=STEP_TOL_SECONDS,
        )
    except Exception as e:
        print(f"SKIP {path}: {e}")
        summary_rows.append(
            {"file": str(path), "out": None, "rows_in": None, "rows_out": None, "error": str(e)}
        )
        continue
    dest = OUT_DIR / out_name
    filtered.to_csv(dest, index=False)
    summary_rows.append(
        {
            "file": path.name,
            "out": str(dest),
            "rows_in": n_in,
            "rows_out": n_out,
            "dropped": n_in - n_out,
            "error": None,
        }
    )
    print(f"OK {path.name}: {n_in:,} → {n_out:,} rows (dropped {n_in - n_out:,}) → {dest.name}")

summary = pd.DataFrame(summary_rows)
summary_path = OUT_DIR / "_filter_summary.csv"
summary.to_csv(summary_path, index=False)
print(f"\nWrote summary: {summary_path}")
try:
    from IPython.display import display

    display(summary)
except Exception:
    print(summary.to_string())

OK AHU_2_9_Blower_DE_A.csv: 61,795 → 58,517 rows (dropped 3,278) → AHU_2_9_Blower_DE_A.csv
OK AHU_2_9_Blower_DE_V.csv: 84,332 → 81,522 rows (dropped 2,810) → AHU_2_9_Blower_DE_V.csv
OK AHU_2_9_Blower_DE_Vibration_X.csv: 149,034 → 133,487 rows (dropped 15,547) → AHU_2_9_Blower_DE_Vibration_X.csv
OK AHU_2_9_Blower_NDE_A.csv: 91,528 → 86,247 rows (dropped 5,281) → AHU_2_9_Blower_NDE_A.csv
OK AHU_2_9_Blower_NDE_H.csv: 56,607 → 41,925 rows (dropped 14,682) → AHU_2_9_Blower_NDE_H.csv
OK AHU_2_9_Blower_NDE_V.csv: 82,917 → 78,211 rows (dropped 4,706) → AHU_2_9_Blower_NDE_V.csv
OK AHU_2_9_motor_DE_H.csv: 64,859 → 58,824 rows (dropped 6,035) → AHU_2_9_motor_DE_H.csv
OK AHU_2_9_motor_NDE_H.csv: 73,402 → 63,992 rows (dropped 9,410) → AHU_2_9_motor_NDE_H.csv

Wrote summary: c:\Users\NGYX\Desktop\Murata_NGYX\data_5mins_frequency\_filter_summary.csv


,file,out,rows_in,rows_out,dropped,error
0,AHU_2_9_Blower_DE_A.csv,c:\Users\NGYX\Desktop\Murata_NGYX\data_5mins_f...,61795,58517,3278,None
1,AHU_2_9_Blower_DE_V.csv,c:\Users\NGYX\Desktop\Murata_NGYX\data_5mins_f...,84332,81522,2810,None
2,AHU_2_9_Blower_DE_Vibration_X.csv,c:\Users\NGYX\Desktop\Murata_NGYX\data_5mins_f...,149034,133487,15547,None
3,AHU_2_9_Blower_NDE_A.csv,c:\Users\NGYX\Desktop\Murata_NGYX\data_5mins_f...,91528,86247,5281,None
4,AHU_2_9_Blower_NDE_H.csv,c:\Users\NGYX\Desktop\Murata_NGYX\data_5mins_f...,56607,41925,14682,None
5,AHU_2_9_Blower_NDE_V.csv,c:\Users\NGYX\Desktop\Murata_NGYX\data_5mins_f...,82917,78211,4706,None
6,AHU_2_9_motor_DE_H.csv,c:\Users\NGYX\Desktop\Murata_NGYX\data_5mins_f...,64859,58824,6035,None
7,AHU_2_9_motor_NDE_H.csv,c:\Users\NGYX\Desktop\Murata_NGYX\data_5mins_f...,73402,63992,9410,None
